# 07-units (Medellin, PM2.5) -- dimensionally constrained search

Same idea as `bogota/07_units_equation_discovery.ipynb`: the pool is the existing same-day no-lag selection (so it is the free-form run's pool), every variable gets a unit, and PySR is restricted to dimensionally homogeneous expressions. Units: gases and PM as mass concentrations (`kg/m^3`, the documented assumption of the units audit, A4), `ventilation_station` = BLH x wind speed (`m^2/s`), `tke_proxy` = 0.5 (var u10 + var v10) (`m^2/s^2`), fire counts dimensionless. Three seeds.

Scoring (same protocol as the Bogota lead equations, notebooks A2/A4): leave-one-year-out, constants refitted by non-linear least squares on the training years, **both** pooled R2 and mean of per-year R2, on the rows the search used. The existing free-form front is scored on the same rows.

In [1]:
# %% Settings (edit here, then Run All)
CITY, TARGET, TAG = 'medellin', 'pm25', 'units_fire2'
MASTER = 'medellin_master_dataset.parquet'                         # file in Data/processed/colombia/<city>/
SELECTION_JSON = 'medellin_feature_selection.json'                    # its selected_no_lag is the pool (identical to the free-form run)
EXTRA_FEATURES = ['no2']             # secondary_pm_proxy = NO2 x radiation is split into raw factors (radiation_mean is already selected)
DROP_FEATURES = ['secondary_pm_proxy']   # removed from it (split into raw factors)
SEEDS = [0, 1, 2]                           # symbolic search is stochastic: report all of them
PYSR_KW = dict(niterations=200, binary_operators=["+", "-", "*", "/"], unary_operators=["log", "sqrt", "exp"],
               model_selection="best", maxsize=25, populations=24)     # identical to the Bogota constrained runs
UNIT_MAP = {'hotspots_sector_NE_lag2': '', 'hotspots_sector_SE_lag2': '', 'hotspots_magdalena_valley': '', 'radiation_mean': 'W/m^2', 'radiation_station_lag1': 'W/m^2', 'no_lag2': 'kg/m^3', 'no2': 'kg/m^3', 'w700_lag3': 'Pa/s'}
Y_UNITS = 'kg/m^3'
FREEFORM_FRONT = 'medellin_pysr_pareto_front.csv'                     # the existing free-form front, scored here on the same rows
MAX_COMPLEXITY_REPORTED = 25

In [2]:
import sys, json, re, warnings
from pathlib import Path
PROJECT_ROOT = Path("/path/to/project")
sys.path.insert(0, str(PROJECT_ROOT / "Scripts"))
import numpy as np, pandas as pd, sympy as sp
from scipy.optimize import curve_fit
from sklearn.metrics import r2_score
DATA_DIR = PROJECT_ROOT / "Data" / "processed" / "colombia" / CITY
df = pd.read_parquet(DATA_DIR / MASTER)
df["date"] = pd.to_datetime(df["date"]); df["year"] = df["date"].dt.year
df = df.dropna(subset=[TARGET]).reset_index(drop=True)
selection = json.load(open(DATA_DIR / SELECTION_JSON))
features = [f for f in selection["selected_no_lag"] if f not in DROP_FEATURES]
for f in EXTRA_FEATURES:
    if f not in features: features.append(f)
print("pool:", features)
data = df.dropna(subset=features + [TARGET]).reset_index(drop=True)
X = data[features].to_numpy(); y = data[TARGET].to_numpy()
X_units = [UNIT_MAP[f] for f in features]
print(data.shape, dict(zip(features, X_units)), "years:", sorted(data.year.unique()))

pool: ['hotspots_sector_NE_lag2', 'radiation_mean', 'hotspots_magdalena_valley', 'no_lag2', 'w700_lag3', 'hotspots_sector_SE_lag2', 'radiation_station_lag1', 'no2']
(2013, 257) {'hotspots_sector_NE_lag2': '', 'radiation_mean': 'W/m^2', 'hotspots_magdalena_valley': '', 'no_lag2': 'kg/m^3', 'w700_lag3': 'Pa/s', 'hotspots_sector_SE_lag2': '', 'radiation_station_lag1': 'W/m^2', 'no2': 'kg/m^3'} years: [np.int32(2015), np.int32(2016), np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022)]


In [3]:
from pysr import PySRRegressor
fronts = {}
for seed in SEEDS:
    sr = PySRRegressor(**PYSR_KW, random_state=seed)
    sr.fit(X, y, variable_names=features, X_units=X_units, y_units=Y_UNITS)
    f = sr.equations_[["complexity", "loss", "equation"]].copy(); f["seed"] = seed
    f.to_csv(DATA_DIR / f"{CITY}_{TARGET}_pysr_{TAG}_pareto_front_seed{seed}.csv", index=False)
    fronts[seed] = f
    print("seed", seed, "done; best:", sr.get_best()["equation"])

Compiling Julia backend...


/path/to/project/.venv/lib/python3.11/site-packages/pysr/sr.py:2385: UserWarning: Note: Setting `random_state` without also setting `deterministic=True` and `parallelism='serial'` will result in non-deterministic searches.
  warnings.warn(


┌ Warning: You are using dimensional constraints, but `dimensional_constraint_penalty` was not set. The default penalty of `1000.0` will be used.
└ @ SymbolicRegression ~/.julia/packages/SymbolicRegression/O7o63/src/Configure.jl:73
[ Info: Started!
[ Info: Final population:
[ Info: Results saved to:



Expressions evaluated per second: 1.010e+04
Progress: 20 / 200 total iterations (10.062%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           4.953e+01  0.000e+00  y[m⁻³ kg] = 18.916[?]
3           4.231e+01  7.871e-02  y[m⁻³ kg] = no2[m⁻³ kg] * 0.50683[?]
4           3.897e+01  8.222e-02  y[m⁻³ kg] = sqrt(no2[m⁻³ kg]) / 0.31289[?]
5           3.458e+01  1.197e-01  y[m⁻³ kg] = (hotspots_sector_NE_lag2 * 0.0080531[?...
                                              ]) + 17.378[?]
6           3.322e+01  4.003e-02  y[m⁻³ kg] = sqrt(hotspots_sector_NE_lag2 * 0.19925...
                                              [?]) + 13.367[?]
7           2.737e+01  1.936e-01  y[m⁻³ kg] = (hotspots_sector_NE_lag2 * 0.01581[?])...
                                               + (no2[m⁻³ kg] * 0.41046[?

seed 0 done; best: (hotspots_sector_NE_lag2 * 0.011168355) + (no2 * 0.43956104)


/path/to/project/.venv/lib/python3.11/site-packages/pysr/sr.py:2385: UserWarning: Note: Setting `random_state` without also setting `deterministic=True` and `parallelism='serial'` will result in non-deterministic searches.
  warnings.warn(



Expressions evaluated per second: 1.210e+04
Progress: 22 / 200 total iterations (11.458%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           4.953e+01  0.000e+00  y[m⁻³ kg] = 18.917[?]
3           4.231e+01  7.873e-02  y[m⁻³ kg] = no2[m⁻³ kg] * 0.50573[?]
4           3.898e+01  8.215e-02  y[m⁻³ kg] = sqrt(no2[m⁻³ kg]) * 3.1979[?]
5           3.375e+01  1.438e-01  y[m⁻³ kg] = 16.719[?] - (hotspots_sector_NE_lag2 *...
                                               -0.010536[?])
7           2.388e+01  1.729e-01  y[m⁻³ kg] = (no2[m⁻³ kg] * 0.446[?]) - (hotspots_s...
                                              ector_NE_lag2 * -0.010554[?])
8           2.207e+01  7.889e-02  y[m⁻³ kg] = (no2[m⁻³ kg] * 0.35358[?]) - (sqrt(hot...
                                              spots_sector_NE

┌ Warning: You are using dimensional constraints, but `dimensional_constraint_penalty` was not set. The default penalty of `1000.0` will be used.
└ @ SymbolicRegression ~/.julia/packages/SymbolicRegression/O7o63/src/Configure.jl:73
[ Info: Started!
[ Info: Final population:
[ Info: Results saved to:
/path/to/project/.venv/lib/python3.11/site-packages/pysr/sr.py:2385: UserWarning: Note: Setting `random_state` without also setting `deterministic=True` and `parallelism='serial'` will result in non-deterministic searches.
  warnings.warn(



Expressions evaluated per second: 1.110e+04
Progress: 21 / 200 total iterations (10.667%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           4.953e+01  0.000e+00  y[m⁻³ kg] = 18.921[?]
3           4.232e+01  7.860e-02  y[m⁻³ kg] = no2[m⁻³ kg] * 0.5086[?]
4           3.897e+01  8.248e-02  y[m⁻³ kg] = sqrt(no2[m⁻³ kg]) * 3.1818[?]
5           3.659e+01  6.311e-02  y[m⁻³ kg] = (hotspots_sector_NE_lag2 / 150.54[?]) ...
                                              + 16.433[?]
6           3.338e+01  9.185e-02  y[m⁻³ kg] = sqrt(hotspots_sector_NE_lag2 * 0.177[?...
                                              ]) + 13.421[?]
8           2.371e+01  1.711e-01  y[m⁻³ kg] = sqrt(((no2[m⁻³ kg] * 16.025[?]) + hots...
                                              pots_sector_NE_lag2) * 0.45507[?])

┌ Warning: You are using dimensional constraints, but `dimensional_constraint_penalty` was not set. The default penalty of `1000.0` will be used.
└ @ SymbolicRegression ~/.julia/packages/SymbolicRegression/O7o63/src/Configure.jl:73
[ Info: Started!
[ Info: Final population:
[ Info: Results saved to:


In [4]:
FLOAT = re.compile(r"(?<![A-Za-z_0-9.])\d+\.\d+(?:[eE][+-]?\d+)?")
def make_model(eq, feats):
    vals = []
    def sub(m): vals.append(float(m.group(0))); return f"c{len(vals)-1}"
    s = FLOAT.sub(sub, eq)
    syms = {f: sp.Symbol(f) for f in feats}; cs = [sp.Symbol(f"c{i}") for i in range(len(vals))]
    expr = sp.sympify(s, locals={**syms, **{str(c): c for c in cs}, "sqrt": sp.sqrt, "log": sp.log, "exp": sp.exp})
    used = [f for f in feats if syms[f] in expr.free_symbols]
    keep = [i for i, c in enumerate(cs) if c in expr.free_symbols]
    fn = sp.lambdify([syms[f] for f in used] + [cs[i] for i in keep], expr, "numpy")
    p0 = np.array([vals[i] for i in keep])
    return (lambda Xt, *p: fn(*Xt, *p)), used, p0

yy = data[TARGET].to_numpy(float); years = data["year"].to_numpy(); YRS = sorted(set(years))
def score(eq, feats):
    model, used, p0 = make_model(eq, feats)
    if not used:                                   # constant equation: leave-one-year-out mean
        oof = np.concatenate([np.full((years == yr).sum(), np.mean(yy[years != yr])) for yr in YRS if (years == yr).sum() >= 30]); yo = np.concatenate([yy[years == yr] for yr in YRS if (years == yr).sum() >= 30])
        per = [r2_score(yy[years == yr], np.full((years == yr).sum(), np.mean(yy[years != yr]))) for yr in YRS if (years == yr).sum() >= 30]
        return dict(r2_insample=0.0, r2_pooled=r2_score(yo, oof), r2_mean_years=float(np.mean(per)), vars="")
    Xt = tuple(data[f].to_numpy(float) for f in used)
    with np.errstate(all="ignore"), warnings.catch_warnings():
        warnings.simplefilter("ignore")
        try: popt, _ = curve_fit(model, Xt, yy, p0=p0, maxfev=20000)
        except Exception: popt = p0
        pred = np.full(len(yy), np.nan)
        for yr in YRS:
            te = years == yr
            try: pf, _ = curve_fit(model, tuple(a[~te] for a in Xt), yy[~te], p0=popt, maxfev=20000)
            except Exception: pf = popt
            pred[te] = model(tuple(a[te] for a in Xt), *pf)
        ok = np.all(np.isfinite(pred)); ins = model(Xt, *popt)
        per = [r2_score(yy[years == yr], pred[years == yr]) for yr in YRS if (years == yr).sum() >= 30] if ok else [np.nan]
        return dict(r2_insample=r2_score(yy, ins) if np.all(np.isfinite(ins)) else np.nan, r2_pooled=r2_score(yy, pred) if ok else np.nan,
                    r2_mean_years=float(np.mean(per)), vars=",".join(used))

rows = []
for seed, f in fronts.items():
    for _, r in f.iterrows():
        if r.complexity <= MAX_COMPLEXITY_REPORTED: rows.append(dict(seed=str(seed), complexity=int(r.complexity), equation=r.equation, **score(r.equation, features)))
ff = pd.read_csv(DATA_DIR / FREEFORM_FRONT).sort_values("complexity")
ff_feats = sorted({t for e in ff.equation for t in re.findall(r"[A-Za-z_][A-Za-z0-9_]*", e) if t in data.columns})
for _, r in ff.iterrows():
    if r.complexity <= MAX_COMPLEXITY_REPORTED:
        try: rows.append(dict(seed="free-form", complexity=int(r.complexity), equation=r.equation, **score(r.equation, ff_feats)))
        except Exception as e: print("free-form eq skipped", r.complexity, e)
scored = pd.DataFrame(rows); scored.to_csv(DATA_DIR / f"{CITY}_{TARGET}_pysr_{TAG}_scored.csv", index=False)
pd.set_option("display.width", 250, "display.max_colwidth", 110)
print("n days", len(data)); print(scored[["seed", "complexity", "r2_pooled", "r2_mean_years", "vars"]].round(3).to_string(index=False))

n days 2013
     seed  complexity  r2_pooled  r2_mean_years                                                                                                        vars
        0           1     -0.093         -0.473                                                                                                            
        0           3      0.128         -0.174                                                                                                         no2
        0           4      0.172         -0.126                                                                                                         no2
        0           5      0.219         -0.072                                                                                     hotspots_sector_NE_lag2
        0           6      0.240         -0.063                                                                                     hotspots_sector_NE_lag2
        0           7      0.510          0.271     